# CineSense-AI: Recommendation Engine

## Objective

This notebook develops the core recommendation engine for CineSense-AI.

Multiple recommendation approaches are implemented and compared, including:

- Popularity-Based Recommendation
- Content-Based Filtering
- Collaborative Filtering
- Hybrid Recommendation

The goal is to combine movie metadata and user behavior to create accurate and personalized movie recommendations.

The core recommendation engine developed here will later support advanced CineSense-AI features such as context-aware recommendations, explainable recommendations, group recommendations, hidden-gem discovery, and diversity-aware ranking.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving movie_genres.csv to movie_genres.csv
Saving movies_final.csv to movies_final.csv
Saving ratings_clean.csv to ratings_clean.csv
Saving tags_clean.csv to tags_clean.csv


In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv("movies_final.csv")
ratings = pd.read_csv("ratings_clean.csv")
tags = pd.read_csv("tags_clean.csv")
movie_genres = pd.read_csv("movie_genres.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Movie Genres:", movie_genres.shape)

Movies: (9742, 9)
Ratings: (100836, 5)
Tags: (3683, 6)
Movie Genres: (22084, 2)


## 1. Popularity-Based Recommendation

A popularity-based baseline recommends movies using both average rating and rating volume.

A weighted rating approach is used to prevent movies with very few ratings from dominating the rankings.

In [ ]:
# Global average rating
C = ratings["rating"].mean()

# Minimum number of ratings required
m = movies["rating_count"].quantile(0.75)

print("Global Average Rating:", round(C, 2))
print("Minimum Rating Threshold:", m)

Global Average Rating: 3.5
Minimum Rating Threshold: 9.0


In [ ]:
qualified_movies = movies[
    movies["rating_count"] >= m
].copy()

qualified_movies["weighted_score"] = (
    (qualified_movies["rating_count"] /
     (qualified_movies["rating_count"] + m))
    * qualified_movies["average_rating"]
    +
    (m /
     (qualified_movies["rating_count"] + m))
    * C
)

popular_recommendations = (
    qualified_movies
    .sort_values(
        "weighted_score",
        ascending=False
    )
    [
        [
            "movieId",
            "clean_title",
            "year",
            "genres",
            "rating_count",
            "average_rating",
            "weighted_score"
        ]
    ]
    .head(10)
)

display(popular_recommendations)

,movieId,clean_title,year,genres,rating_count,average_rating,weighted_score
277,318,"Shawshank Redemption, The",1994.0,Crime|Drama,317,4.43,4.404368
659,858,"Godfather, The",1972.0,Crime|Drama,192,4.29,4.254697
2226,2959,Fight Club,1999.0,Action|Crime|Drama|Thriller,218,4.27,4.239533
922,1221,"Godfather: Part II, The",1974.0,Crime|Drama,129,4.26,4.210536
46,50,"Usual Suspects, The",1995.0,Crime|Mystery|Thriller,204,4.24,4.208798
224,260,Star Wars: Episode IV - A New Hope,1977.0,Action|Adventure|Sci-Fi,251,4.23,4.204785
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,97,4.27,4.204755
914,1213,Goodfellas,1990.0,Crime|Drama,126,4.25,4.200104
6710,58559,"Dark Knight, The",2008.0,Action|Crime|Drama|IMAX,149,4.24,4.197937
6315,48516,"Departed, The",2006.0,Crime|Drama|Thriller,107,4.25,4.191931


In [ ]:
def recommend_popular_movies(movies_df, top_n=10, genre=None):

    data = movies_df.copy()

    # Optional genre filtering
    if genre:
        data = data[
            data["genres"].str.contains(
                genre,
                case=False,
                na=False
            )
        ]

    # Minimum popularity threshold
    min_ratings = data["rating_count"].quantile(0.75)

    qualified = data[
        data["rating_count"] >= min_ratings
    ].copy()

    # Weighted rating
    qualified["weighted_score"] = (
        (qualified["rating_count"] /
         (qualified["rating_count"] + min_ratings))
        * qualified["average_rating"]
        +
        (min_ratings /
         (qualified["rating_count"] + min_ratings))
        * C
    )

    return (
        qualified
        .sort_values(
            "weighted_score",
            ascending=False
        )
        [
            [
                "clean_title",
                "year",
                "genres",
                "rating_count",
                "average_rating",
                "weighted_score"
            ]
        ]
        .head(top_n)
    )

In [ ]:
recommend_popular_movies(
    movies,
    top_n=10
)

,clean_title,year,genres,rating_count,average_rating,weighted_score
277,"Shawshank Redemption, The",1994.0,Crime|Drama,317,4.43,4.404368
659,"Godfather, The",1972.0,Crime|Drama,192,4.29,4.254697
2226,Fight Club,1999.0,Action|Crime|Drama|Thriller,218,4.27,4.239533
922,"Godfather: Part II, The",1974.0,Crime|Drama,129,4.26,4.210536
46,"Usual Suspects, The",1995.0,Crime|Mystery|Thriller,204,4.24,4.208798
224,Star Wars: Episode IV - A New Hope,1977.0,Action|Adventure|Sci-Fi,251,4.23,4.204785
602,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,97,4.27,4.204755
914,Goodfellas,1990.0,Crime|Drama,126,4.25,4.200104
6710,"Dark Knight, The",2008.0,Action|Crime|Drama|IMAX,149,4.24,4.197937
6315,"Departed, The",2006.0,Crime|Drama|Thriller,107,4.25,4.191931


In [ ]:
recommend_popular_movies(
    movies,
    top_n=10,
    genre="Comedy"
)

,clean_title,year,genres,rating_count,average_rating,weighted_score
602,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,97,4.27,4.204755
899,"Princess Bride, The",1987.0,Action|Adventure|Comedy|Fantasy|Romance,142,4.23,4.186583
257,Pulp Fiction,1994.0,Comedy|Crime|Drama|Thriller,307,4.20,4.180108
314,Forrest Gump,1994.0,Comedy|Drama|Romance|War,329,4.16,4.142467
3622,"Amelie (Fabuleux destin d'Amélie Poulain, Le)",2001.0,Comedy|Romance,120,4.18,4.132667
863,Monty Python and the Holy Grail,1975.0,Adventure|Comedy|Fantasy,136,4.16,4.119131
680,"Philadelphia Story, The",1940.0,Comedy|Drama|Romance,29,4.31,4.118527
2996,Snatch,2000.0,Comedy|Crime|Thriller,93,4.16,4.101902
520,Fargo,1996.0,Comedy|Crime|Drama|Thriller,181,4.12,4.090705
1730,Life Is Beautiful (La Vita è bella),1997.0,Comedy|Drama|Romance|War,88,4.15,4.089835


## 2. Content-Based Recommendation

Content-based filtering recommends movies that are similar based on their characteristics.

For the initial model, movie genres and user-generated tags are combined into a single content representation. TF-IDF vectorization and cosine similarity are then used to measure similarity between movies.

In [ ]:
# Prepare content features

movies_content = movies.copy()

movies_content["genres_text"] = (
    movies_content["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

movies_content["combined_tags"] = (
    movies_content["combined_tags"]
    .fillna("")
    .astype(str)
    .str.lower()
)

movies_content["content_features"] = (
    movies_content["genres_text"]
    + " "
    + movies_content["combined_tags"]
)

display(
    movies_content[
        [
            "clean_title",
            "genres_text",
            "combined_tags",
            "content_features"
        ]
    ].head()
)

,clean_title,genres_text,combined_tags,content_features
0,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,adventure animation children comedy fantasy pi...
1,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,adventure children fantasy fantasy magic board...
2,Grumpier Old Men,comedy romance,moldy old,comedy romance moldy old
3,Waiting to Exhale,comedy drama romance,,comedy drama romance
4,Father of the Bride Part II,comedy,pregnancy remake,comedy pregnancy remake


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies_content["content_features"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (9742, 1677)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_similar_movies(
    movie_title,
    movies_df,
    tfidf_matrix,
    top_n=10
):

    # Find matching movie
    matches = movies_df[
        movies_df["clean_title"].str.lower()
        == movie_title.lower()
    ]

    if matches.empty:
        return "Movie not found."

    # Use first matching movie
    movie_index = matches.index[0]

    # Calculate similarity against all movies
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    # Sort by similarity
    similar_indices = (
        similarity_scores
        .argsort()[::-1]
    )

    # Remove the input movie itself
    similar_indices = [
        idx
        for idx in similar_indices
        if idx != movie_index
    ][:top_n]

    recommendations = movies_df.iloc[
        similar_indices
    ].copy()

    recommendations["similarity_score"] = (
        similarity_scores[
            similar_indices
        ]
    )

    return recommendations[
        [
            "clean_title",
            "year",
            "genres",
            "average_rating",
            "rating_count",
            "similarity_score"
        ]
    ]

In [ ]:
recommend_similar_movies(
    "Toy Story",
    movies_content,
    tfidf_matrix,
    top_n=10
)

,clean_title,year,genres,average_rating,rating_count,similarity_score
1757,"Bug's Life, A",1998.0,Adventure|Animation|Children|Comedy,3.52,92,0.862225
2355,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,3.86,97,0.644038
8695,Guardians of the Galaxy 2,2017.0,Action|Adventure|Sci-Fi,3.93,27,0.367650
8219,Turbo,2013.0,Adventure|Animation|Children|Comedy|Fantasy,2.50,1,0.357912
8927,The Good Dinosaur,2015.0,Adventure|Animation|Children|Comedy|Fantasy,3.00,4,0.357912
3000,"Emperor's New Groove, The",2000.0,Adventure|Animation|Children|Comedy|Fantasy,3.72,37,0.357912
9430,Moana,2016.0,Adventure|Animation|Children|Comedy|Fantasy,3.45,10,0.357912
6486,Shrek the Third,2007.0,Adventure|Animation|Children|Comedy|Fantasy,3.02,21,0.357912
6194,"Wild, The",2006.0,Adventure|Animation|Children|Comedy|Fantasy,2.50,1,0.357912
6948,"Tale of Despereaux, The",2008.0,Adventure|Animation|Children|Comedy|Fantasy,3.00,1,0.357912


In [ ]:
recommend_similar_movies(
    "The Dark Knight",
    movies_content,
    tfidf_matrix,
    top_n=10
)

'Movie not found.'

In [ ]:
movies_content[
    movies_content["clean_title"]
    .str.contains(
        "Dark Knight",
        case=False,
        na=False
    )
][["clean_title", "year"]]

,clean_title,year
6710,"Dark Knight, The",2008.0
7768,"Dark Knight Rises, The",2012.0
8032,"Batman: The Dark Knight Returns, Part 1",2012.0
8080,"Batman: The Dark Knight Returns, Part 2",2013.0


In [ ]:
!pip install scikit-surprise -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.3 MB/s eta 0:00:00


In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

## 3. Collaborative Filtering

Collaborative filtering generates personalized recommendations by learning patterns from user-movie interactions.

An SVD-based matrix factorization model is used to learn latent relationships between users and movies from historical rating data.

In [ ]:
# Define rating scale
reader = Reader(
    rating_scale=(
        ratings["rating"].min(),
        ratings["rating"].max()
    )
)

# Load ratings into Surprise format
data = Dataset.load_from_df(
    ratings[
        ["userId", "movieId", "rating"]
    ],
    reader
)

print("Collaborative filtering dataset prepared successfully.")

Collaborative filtering dataset prepared successfully.


In [ ]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

print(
    "Training ratings:",
    trainset.n_ratings
)

print(
    "Testing ratings:",
    len(testset)
)

Training ratings: 80668
Testing ratings: 20168


In [ ]:
svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    random_state=42
)

svd_model.fit(trainset)

print("SVD collaborative filtering model trained successfully!")

SVD collaborative filtering model trained successfully!


In [ ]:
predictions = svd_model.test(
    testset
)

In [ ]:
rmse = accuracy.rmse(
    predictions,
    verbose=False
)

mae = accuracy.mae(
    predictions,
    verbose=False
)

print(
    "RMSE:",
    round(rmse, 4)
)

print(
    "MAE:",
    round(mae, 4)
)

RMSE: 0.8807
MAE: 0.6766


### Model Evaluation

The collaborative filtering model is evaluated using:

- **RMSE (Root Mean Squared Error):** Measures the magnitude of rating prediction errors while penalizing larger errors.
- **MAE (Mean Absolute Error):** Measures the average absolute difference between predicted and actual ratings.

Lower RMSE and MAE values indicate better rating prediction performance.

In [ ]:
user_id = 1
movie_id = 1

prediction = svd_model.predict(
    user_id,
    movie_id
)

print(
    "User:",
    user_id
)

print(
    "Movie:",
    movie_id
)

print(
    "Predicted Rating:",
    round(
        prediction.est,
        2
    )
)

User: 1
Movie: 1
Predicted Rating: 4.57


In [ ]:
def recommend_for_user(
    user_id,
    model,
    ratings_df,
    movies_df,
    top_n=10
):

    # Movies already rated by user
    rated_movies = set(
        ratings_df[
            ratings_df["userId"] == user_id
        ]["movieId"]
    )

    # Movies not yet rated by user
    unseen_movies = movies_df[
        ~movies_df["movieId"].isin(
            rated_movies
        )
    ].copy()

    # Predict ratings
    predicted_ratings = []

    for movie_id in unseen_movies["movieId"]:

        prediction = model.predict(
            user_id,
            movie_id
        )

        predicted_ratings.append(
            prediction.est
        )

    unseen_movies[
        "predicted_rating"
    ] = predicted_ratings

    recommendations = (
        unseen_movies
        .sort_values(
            "predicted_rating",
            ascending=False
        )
        .head(top_n)
    )

    return recommendations[
        [
            "movieId",
            "clean_title",
            "year",
            "genres",
            "average_rating",
            "rating_count",
            "predicted_rating"
        ]
    ]

In [ ]:
recommend_for_user(
    user_id=1,
    model=svd_model,
    ratings_df=ratings,
    movies_df=movies,
    top_n=10
)

,movieId,clean_title,year,genres,average_rating,rating_count,predicted_rating
474,541,Blade Runner,1982.0,Action|Sci-Fi|Thriller,4.10,124,5.0
6315,48516,"Departed, The",2006.0,Crime|Drama|Thriller,4.25,107,5.0
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,4.27,97,5.0
878,1172,Cinema Paradiso (Nuovo cinema Paradiso),1989.0,Drama,4.16,34,5.0
1422,1945,On the Waterfront,1954.0,Crime|Drama,4.19,24,5.0
596,741,Ghost in the Shell (Kôkaku kidôtai),1995.0,Animation|Sci-Fi,4.15,27,5.0
1494,2019,Seven Samurai (Shichinin no samurai),1954.0,Action|Adventure|Drama,4.19,48,5.0
694,912,Casablanca,1942.0,Drama|Romance,4.24,100,5.0
690,908,North by Northwest,1959.0,Action|Adventure|Mystery|Romance|Thriller,4.18,57,5.0
841,1104,"Streetcar Named Desire, A",1951.0,Drama,4.47,20,5.0


In [ ]:
recommend_for_user(
    user_id=50,
    model=svd_model,
    ratings_df=ratings,
    movies_df=movies,
    top_n=10
)

,movieId,clean_title,year,genres,average_rating,rating_count,predicted_rating
2020,2692,Run Lola Run (Lola rennt),1998.0,Action|Crime,4.00,75,3.598210
585,720,Wallace & Gromit: The Best of Aardman Animation,1996.0,Adventure|Animation|Comedy,4.09,27,3.573316
3622,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le)",2001.0,Comedy|Romance,4.18,120,3.570400
6315,48516,"Departed, The",2006.0,Crime|Drama|Thriller,4.25,107,3.558971
2765,3703,"Road Warrior, The (Mad Max 2)",1981.0,Action|Adventure|Sci-Fi|Thriller,4.04,40,3.557193
97,110,Braveheart,1995.0,Action|Drama|War,4.03,237,3.554123
692,910,Some Like It Hot,1959.0,Comedy|Crime,4.01,50,3.550953
1711,2300,"Producers, The",1968.0,Comedy,3.97,33,3.532612
943,1244,Manhattan,1979.0,Comedy|Drama|Romance,4.11,33,3.528812
4909,7361,Eternal Sunshine of the Spotless Mind,2004.0,Drama|Romance|Sci-Fi,4.16,131,3.528801


In [ ]:
def get_user_favorites(
    user_id,
    ratings_df,
    movies_df,
    min_rating=4.0,
    top_n=10
):

    user_ratings = ratings_df[
        (ratings_df["userId"] == user_id)
        &
        (ratings_df["rating"] >= min_rating)
    ]

    favorites = (
        user_ratings
        .merge(
            movies_df,
            on="movieId",
            how="left"
        )
        .sort_values(
            "rating",
            ascending=False
        )
    )

    return favorites[
        [
            "clean_title",
            "year",
            "genres",
            "rating"
        ]
    ].head(top_n)

In [ ]:
get_user_favorites(
    user_id=1,
    ratings_df=ratings,
    movies_df=movies
)

,clean_title,year,genres,rating
3,Seven (a.k.a. Se7en),1995.0,Mystery|Thriller,5.0
7,Rob Roy,1995.0,Action|Drama|Romance|War,5.0
5,Bottle Rocket,1996.0,Adventure|Comedy|Crime|Romance,5.0
4,"Usual Suspects, The",1995.0,Crime|Mystery|Thriller,5.0
11,Dumb & Dumber (Dumb and Dumber),1994.0,Adventure|Comedy,5.0
10,Billy Madison,1995.0,Comedy,5.0
9,Desperado,1995.0,Action|Romance|Western,5.0
8,Canadian Bacon,1995.0,Comedy|War,5.0
68,"Terminator, The",1984.0,Action|Sci-Fi|Thriller,5.0
69,Duck Soup,1933.0,Comedy|Musical|War,5.0


In [ ]:
recommend_for_user(
    user_id=1,
    model=svd_model,
    ratings_df=ratings,
    movies_df=movies
)

,movieId,clean_title,year,genres,average_rating,rating_count,predicted_rating
474,541,Blade Runner,1982.0,Action|Sci-Fi|Thriller,4.10,124,5.0
6315,48516,"Departed, The",2006.0,Crime|Drama|Thriller,4.25,107,5.0
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,1964.0,Comedy|War,4.27,97,5.0
878,1172,Cinema Paradiso (Nuovo cinema Paradiso),1989.0,Drama,4.16,34,5.0
1422,1945,On the Waterfront,1954.0,Crime|Drama,4.19,24,5.0
596,741,Ghost in the Shell (Kôkaku kidôtai),1995.0,Animation|Sci-Fi,4.15,27,5.0
1494,2019,Seven Samurai (Shichinin no samurai),1954.0,Action|Adventure|Drama,4.19,48,5.0
694,912,Casablanca,1942.0,Drama|Romance,4.24,100,5.0
690,908,North by Northwest,1959.0,Action|Adventure|Mystery|Romance|Thriller,4.18,57,5.0
841,1104,"Streetcar Named Desire, A",1951.0,Drama,4.47,20,5.0


## 4. Hybrid Recommendation System

The hybrid recommendation model combines multiple recommendation signals:

- Collaborative filtering prediction
- Content similarity
- Movie quality

Combining these signals helps reduce the limitations of relying on a single recommendation approach.

The initial weights are experimental and will be refined during evaluation.

In [ ]:
def hybrid_recommend(
    user_id,
    liked_movie,
    model,
    ratings_df,
    movies_df,
    tfidf_matrix,
    top_n=10
):

    # Find selected movie
    matches = movies_df[
        movies_df["clean_title"].str.lower()
        == liked_movie.lower()
    ]

    if matches.empty:
        return "Movie not found."

    liked_index = matches.index[0]

    # Content similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[liked_index],
        tfidf_matrix
    ).flatten()

    # Movies already rated
    rated_movies = set(
        ratings_df[
            ratings_df["userId"] == user_id
        ]["movieId"]
    )

    candidates = movies_df[
        ~movies_df["movieId"].isin(
            rated_movies
        )
    ].copy()

    # Content score
    candidates[
        "content_score"
    ] = similarity_scores[
        candidates.index
    ]

    # Collaborative score
    candidates[
        "collaborative_score"
    ] = candidates[
        "movieId"
    ].apply(
        lambda movie_id:
        model.predict(
            user_id,
            movie_id
        ).est / 5
    )

    # Quality score
    candidates[
        "quality_score"
    ] = (
        candidates[
            "average_rating"
        ] / 5
    )

    # Hybrid score
    candidates[
        "hybrid_score"
    ] = (
        0.50
        * candidates[
            "collaborative_score"
        ]
        +
        0.30
        * candidates[
            "content_score"
        ]
        +
        0.20
        * candidates[
            "quality_score"
        ]
    )

    recommendations = (
        candidates
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
    )

    return recommendations[
        [
            "clean_title",
            "year",
            "genres",
            "average_rating",
            "rating_count",
            "collaborative_score",
            "content_score",
            "hybrid_score"
        ]
    ]

In [ ]:
hybrid_recommend(
    user_id=1,
    liked_movie="Toy Story",
    model=svd_model,
    ratings_df=ratings,
    movies_df=movies_content,
    tfidf_matrix=tfidf_matrix,
    top_n=10
)

,clean_title,year,genres,average_rating,rating_count,collaborative_score,content_score,hybrid_score
1757,"Bug's Life, A",1998.0,Adventure|Animation|Children|Comedy,3.52,92,0.856269,0.862225,0.827602
2355,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,3.86,97,0.937744,0.644038,0.816483
8707,"Snowflake, the White Gorilla",2011.0,Adventure|Animation|Children|Comedy,5.00,1,0.898309,0.313323,0.743151
7917,Presto,2008.0,Animation|Children|Comedy|Fantasy,5.00,1,0.886092,0.325317,0.740641
7039,Up,2009.0,Adventure|Animation|Children|Drama,4.00,105,0.966047,0.320916,0.739298
9255,Ice Age: The Great Egg-Scapade,2016.0,Adventure|Animation|Children|Comedy,5.00,1,0.883228,0.313323,0.735611
9536,Last Year's Snow Was Falling,1983.0,Animation|Children|Comedy|Fantasy,5.00,1,0.872463,0.325317,0.733827
9583,Gena the Crocodile,1969.0,Animation|Children,5.00,1,0.905628,0.258322,0.730311
8674,Stuart Little 3: Call of the Wild,2005.0,Animation|Children|Comedy|Fantasy,5.00,1,0.864574,0.325317,0.729882
9586,"In the blue sea, in the white foam.",1984.0,Animation|Children|Fantasy,5.00,1,0.868512,0.310901,0.727526
